In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.getOrCreate()

customers = spark.createDataFrame([
    (1,"John ","Hyderabad"),
    (2,"Alice","Chennai"),
    (3,None,"Bangalore")
],["customer_id","name","city"]  )

cars = spark.createDataFrame([
    (101,"Toyota","Camry",30000),
    (102,"Honda","Civic",20000),
    (103,"Hyundai","i20",15000)
],["car_id","brand","model","price"])

sales = spark.createDataFrame([
    (1,1,101,"2024-01-01",1),
    (2,2,102,"2024-01-02",2),
    (3,99,103,"2024-01-03",1)
],["sale_id","customer_id","car_id","sale_date","quantity"])

sales = sales.withColumn("sale_date", to_date(col("sale_date")))

In [0]:
customers.show()
cars.show()
sales.show()

+-----------+-----+---------+
|customer_id| name|     city|
+-----------+-----+---------+
|          1|John |Hyderabad|
|          2|Alice|  Chennai|
|          3| NULL|Bangalore|
+-----------+-----+---------+

+------+-------+-----+------+
|car_id|  brand|model| price|
+------+-------+-----+------+
|   101| Toyota|Camry| 30000|
|   102|  Honda|Civic|-20000|
|   103|Hyundai|  i20| 15000|
+------+-------+-----+------+

+-------+-----------+------+----------+--------+
|sale_id|customer_id|car_id| sale_date|quantity|
+-------+-----------+------+----------+--------+
|      1|          1|   101|2024-01-01|       1|
|      2|          2|   102|2024-01-02|       2|
|      3|         99|   103|2024-01-03|       1|
+-------+-----------+------+----------+--------+



In [0]:
customers.fillna("Unknown").show()

+-----------+-------+---------+
|customer_id|   name|     city|
+-----------+-------+---------+
|          1|  John |Hyderabad|
|          2|  Alice|  Chennai|
|          3|Unknown|Bangalore|
+-----------+-------+---------+



In [0]:
cars=cars.withColumn("Data Validation",
                when(col("price")<=0,"Invalid").otherwise("Valid"))
        

In [0]:
cars.show()

+------+-------+-----+------+---------------+
|car_id|  brand|model| price|Data Validation|
+------+-------+-----+------+---------------+
|   101| Toyota|Camry| 30000|          Valid|
|   102|  Honda|Civic|-20000|        Invalid|
|   103|Hyundai|  i20| 15000|          Valid|
+------+-------+-----+------+---------------+



In [0]:
joined_df=customers.join(sales,on="customer_id",how="right")
joined_df=joined_df.withColumn("Data Validation",when(col("name").isNull(),"Invalid").otherwise("Valid"))
sales=joined_df.drop("name","city")
sales.show()

+-----------+-------+------+----------+--------+---------------+
|customer_id|sale_id|car_id| sale_date|quantity|Data Validation|
+-----------+-------+------+----------+--------+---------------+
|          1|      1|   101|2024-01-01|       1|          Valid|
|          2|      2|   102|2024-01-02|       2|          Valid|
|         99|      3|   103|2024-01-03|       1|        Invalid|
+-----------+-------+------+----------+--------+---------------+



In [0]:
valid_sales=sales.join(customers,on="customer_id",how="inner")
valid_sales.show()

+-----------+-------+------+----------+--------+-----+---------+
|customer_id|sale_id|car_id| sale_date|quantity| name|     city|
+-----------+-------+------+----------+--------+-----+---------+
|          1|      1|   101|2024-01-01|       1|John |Hyderabad|
|          2|      2|   102|2024-01-02|       2|Alice|  Chennai|
+-----------+-------+------+----------+--------+-----+---------+



In [0]:
clean_df=valid_sales.join(cars,on="car_id",how="inner")
clean_df.show()

+------+-----------+-------+----------+--------+-----+---------+------+-----+-----+
|car_id|customer_id|sale_id| sale_date|quantity| name|     city| brand|model|price|
+------+-----------+-------+----------+--------+-----+---------+------+-----+-----+
|   101|          1|      1|2024-01-01|       1|John |Hyderabad|Toyota|Camry|30000|
|   102|          2|      2|2024-01-02|       2|Alice|  Chennai| Honda|Civic|20000|
+------+-----------+-------+----------+--------+-----+---------+------+-----+-----+



In [0]:
clean_df=clean_df.withColumn("revenue",(col("price")*col("quantity")))
clean_df.show()

+------+-----------+-------+----------+--------+-----+---------+------+-----+-----+-------+
|car_id|customer_id|sale_id| sale_date|quantity| name|     city| brand|model|price|revenue|
+------+-----------+-------+----------+--------+-----+---------+------+-----+-----+-------+
|   101|          1|      1|2024-01-01|       1|John |Hyderabad|Toyota|Camry|30000|  30000|
|   102|          2|      2|2024-01-02|       2|Alice|  Chennai| Honda|Civic|20000|  40000|
+------+-----------+-------+----------+--------+-----+---------+------+-----+-----+-------+



In [0]:
clean_df.groupBy("brand").agg(
    count(col("car_id")).alias("No.of cars per brand")
).display()

brand,No.of cars per brand
Toyota,1
Honda,1


In [0]:
clean_df.groupBy("city").agg(
    sum(col("revenue")).alias("City Wise Revenue")
).display()

city,City Wise Revenue
Hyderabad,30000
Chennai,40000


In [0]:
from pyspark.sql.window import Window
window=Window.partitionBy("city").orderBy(col("revenue").desc())
clean_df.select("customer_id","name","city","revenue",rank().over(window).alias("Rank")).filter(col("Rank")<=3).display()

customer_id,name,city,revenue,Rank
2,Alice,Chennai,40000,1
1,John,Hyderabad,30000,1


In [0]:
clean_df.groupBy("customer_id","name").agg(
    count(col("sale_id")).alias("Customers")
).filter(col("Customers")>1).display()

customer_id,name,Customers


In [0]:
clean_df.groupBy(month(col("sale_date"))).agg(
    sum(col("revenue")).alias("Monthly revenue"),
    avg(col("revenue")).alias("Avg revenue"),
    count(col("sale_id")).alias("No.of sales")
).display()

month(sale_date),Monthly revenue,Avg revenue,No.of sales
1,70000,35000.0,2
